# Popularity / Exposure Bias Diagnosis — NingJing

**Author:** NingJing (Group 1, JDSearch)

## This notebook

1. Diagnoses whether the team's final ranking models (BST, LightGBM) are biased toward popular products relative to the NMF/NCF baselines
2. Checks whether the bias comes from model architecture or from shared features (by comparing a black-box model against a white-box model), tests two inference-time fixes for cold items, and traces the root cause.

## Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("/content/drive/MyDrive/BT4222 Final/JDSearch")
OUTPUT_DIR = BASE / "processed"

# Loads the lightweight test-split-only checkpoint saved at the end of
# step_2_query_popularity_baseline_models.ipynb (test rows, baseline/popularity
# features, and NMF/NCF hybrid scores -- already filtered to split == 'test'
# and trimmed to only the columns this notebook needs).
test_df = pd.read_parquet(
    OUTPUT_DIR / "ningjing_test_features_with_baselines.parquet"
)
print("Loaded test_df:", test_df.shape)

Mounted at /content/drive
Loaded test_df: (1531525, 14)


## Test-split view

`test_df` above is already the test-split rows (filtered when it was saved in the baseline/feature notebook), so no further filtering is needed here.

## Popularity / Exposure Bias Diagnosis

Comparing the team's final models (BST, LightGBM) against the NMF/NCF baselines on the test split, to check whether the final models are biased toward popular items.

In [2]:
#separate products into buckets according to popularity
test_df['pop_bucket'] = pd.cut(
    test_df['item_pop_train'],
    bins=[-0.1, 0, 1, 5, 20, test_df['item_pop_train'].max()],
    labels=['0 (no history)', '1', '2-5', '6-20', '20+']
)
overall_dist = test_df['pop_bucket'].value_counts(normalize=True).sort_index()
print(overall_dist)

pop_bucket
0 (no history)    0.894772
1                 0.057176
2-5               0.033577
6-20              0.011344
20+               0.003131
Name: proportion, dtype: float64


In [ ]:
#using BST model from "step_6_build_train_BST_model.ipynb"
bst_test = pd.read_parquet("/content/drive/MyDrive/BT4222 Final/JDSearch/processed/bst_literal/test_predictions.parquet")
bst_test = bst_test.rename(columns={"wid": "item_id", "score": "bst_score"})
bst_test['item_id'] = bst_test['item_id'].astype(str)  # align dtype with test_df's item_id (string)

test_df = test_df.merge(
    bst_test[['session_id', 'item_id', 'bst_score']],
    on=['session_id', 'item_id'],
    how='left'
)
print(test_df['bst_score'].isna().mean(), "fraction with no bst_score match (should be close to 0)")

0.0 fraction with no bst_score match (should be close to 0)


In [ ]:
#using LightGBM model's test-split predictions from "step_5_build_run_lightgbm_model.ipynb"
lgbm_test = pd.read_parquet(
    "/content/drive/MyDrive/BT4222 Final/JDSearch/model_artifacts/lightgbm_popularity_query_search/full/test_predictions.parquet"
)
lgbm_test = lgbm_test.rename(columns={"wid": "item_id", "score": "lgbm_score"})
lgbm_test['item_id'] = lgbm_test['item_id'].astype(str)

test_df = test_df.merge(
    lgbm_test[['session_id', 'item_id', 'lgbm_score']],
    on=['session_id', 'item_id'],
    how='left'
)
print(test_df['lgbm_score'].isna().mean(), "fraction with no lgbm_score match (should be close to 0)")

0.0 fraction with no lgbm_score match (should be close to 0)


### Check 1: is each model's top-10 biased toward popular products?

In [5]:
def top10_pop_distribution(df, score_col):
    d = df[['session_id', score_col, 'pop_bucket']].copy()
    np.random.seed(42)
    d['tiebreak'] = np.random.rand(len(d))
    d = d.sort_values(['session_id', score_col, 'tiebreak'], ascending=[True, False, False])
    d['rank'] = d.groupby('session_id').cumcount() + 1
    top10 = d[d['rank'] <= 10]
    return top10['pop_bucket'].value_counts(normalize=True).sort_index()

compare = pd.DataFrame({
    'overall_candidates': overall_dist,
    'nmf_top10': top10_pop_distribution(test_df, 'hybrid_score'),
    'ncf_top10': top10_pop_distribution(test_df, 'ncf_hybrid_score'),
    'bst_top10': top10_pop_distribution(test_df, 'bst_score'),
    'lgbm_top10': top10_pop_distribution(test_df, 'lgbm_score'),
})
print(compare)

                overall_candidates  nmf_top10  ncf_top10  bst_top10  \
pop_bucket                                                            
0 (no history)            0.894772   0.582433   0.574430   0.509429   
1                         0.057176   0.166639   0.166999   0.192875   
2-5                       0.033577   0.152814   0.156462   0.187825   
6-20                      0.011344   0.074548   0.077023   0.084071   
20+                       0.003131   0.023567   0.025087   0.025800   

                lgbm_top10  
pop_bucket                  
0 (no history)    0.513171  
1                 0.196393  
2-5               0.183328  
6-20              0.082002  
20+               0.025105  


### Check 2: how many non-popular products does each model miss (fail to rank in the top 10)?

In [6]:
def recall_at_10_by_bucket(df, score_col):
    d = df[['session_id', score_col, 'label', 'pop_bucket']].copy()
    np.random.seed(42)
    d['tiebreak'] = np.random.rand(len(d))
    d = d.sort_values(['session_id', score_col, 'tiebreak'], ascending=[True, False, False])
    d['rank'] = d.groupby('session_id').cumcount() + 1
    relevant = d[d['label'] > 0].copy()
    relevant['hit_top10'] = relevant['rank'] <= 10
    return relevant.groupby('pop_bucket')['hit_top10'].mean()
print("Popularity bias / cold-start recall, by model:")
print("\nNMF:");   print(recall_at_10_by_bucket(test_df, 'hybrid_score'))
print("\nNCF:");   print(recall_at_10_by_bucket(test_df, 'ncf_hybrid_score'))
print("\nBST:");   print(recall_at_10_by_bucket(test_df, 'bst_score'))
print("\nLightGBM:"); print(recall_at_10_by_bucket(test_df, 'lgbm_score'))

Popularity bias / cold-start recall, by model:

NMF:


/tmp/ipykernel_501/2206598482.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return relevant.groupby('pop_bucket')['hit_top10'].mean()


pop_bucket
0 (no history)    0.227906
1                 0.590541
2-5               0.711589
6-20              0.860918
20+               0.931223
Name: hit_top10, dtype: float64

NCF:


/tmp/ipykernel_501/2206598482.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return relevant.groupby('pop_bucket')['hit_top10'].mean()


pop_bucket
0 (no history)    0.225111
1                 0.577297
2-5               0.729942
6-20              0.877144
20+               0.969432
Name: hit_top10, dtype: float64

BST:


/tmp/ipykernel_501/2206598482.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return relevant.groupby('pop_bucket')['hit_top10'].mean()


pop_bucket
0 (no history)    0.287072
1                 0.662973
2-5               0.839539
6-20              0.956885
20+               0.984716
Name: hit_top10, dtype: float64

LightGBM:
pop_bucket
0 (no history)    0.318612
1                 0.701622
2-5               0.839276
6-20              0.950858
20+               0.980349
Name: hit_top10, dtype: float64


/tmp/ipykernel_501/2206598482.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return relevant.groupby('pop_bucket')['hit_top10'].mean()


## Attempted fix 1: blend in the query-match signal for cold items

For items with zero training-split history, blend the model's own score with the query-match signal, instead of trusting the model score alone.

In [7]:
# Build a cold-start fallback score for BST and LightGBM:
# when an item has training-split history, trust the model's own score;
# when it has zero history (cold item), blend in the query-match signal
# so the ranking is not driven purely by popularity.
COLD_MASK = test_df['pop_bucket'] == '0 (no history)'

def add_fallback_score(df, model_score_col, fallback_score_col):
    out = df[model_score_col].copy()
    # Rank model scores and query-match counts onto a comparable 0-1 scale
    # so blending doesn't let one signal dominate purely by magnitude.
    model_rank = df[model_score_col].rank(pct=True)
    match_rank = df['query_match_count'].rank(pct=True)
    blended = np.where(COLD_MASK, 0.5 * model_rank + 0.5 * match_rank, model_rank)
    return blended

test_df['bst_fallback_score'] = add_fallback_score(test_df, 'bst_score', 'query_match_count')
test_df['lgbm_fallback_score'] = add_fallback_score(test_df, 'lgbm_score', 'query_match_count')

In [8]:
print("Cold-start recall, before vs after fallback blending:")
print("\nBST (original):");          print(recall_at_10_by_bucket(test_df, 'bst_score'))
print("\nBST (with fallback):");      print(recall_at_10_by_bucket(test_df, 'bst_fallback_score'))
print("\nLightGBM (original):");      print(recall_at_10_by_bucket(test_df, 'lgbm_score'))
print("\nLightGBM (with fallback):"); print(recall_at_10_by_bucket(test_df, 'lgbm_fallback_score'))

Cold-start recall, before vs after fallback blending:

BST (original):


/tmp/ipykernel_501/2206598482.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return relevant.groupby('pop_bucket')['hit_top10'].mean()


pop_bucket
0 (no history)    0.287072
1                 0.662973
2-5               0.839539
6-20              0.956885
20+               0.984716
Name: hit_top10, dtype: float64

BST (with fallback):


/tmp/ipykernel_501/2206598482.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return relevant.groupby('pop_bucket')['hit_top10'].mean()


pop_bucket
0 (no history)    0.251945
1                 0.720541
2-5               0.857630
6-20              0.957348
20+               0.985808
Name: hit_top10, dtype: float64

LightGBM (original):


/tmp/ipykernel_501/2206598482.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return relevant.groupby('pop_bucket')['hit_top10'].mean()


pop_bucket
0 (no history)    0.318612
1                 0.701622
2-5               0.839276
6-20              0.950858
20+               0.980349
Name: hit_top10, dtype: float64

LightGBM (with fallback):
pop_bucket
0 (no history)    0.275798
1                 0.758919
2-5               0.859465
6-20              0.956885
20+               0.982533
Name: hit_top10, dtype: float64


/tmp/ipykernel_501/2206598482.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return relevant.groupby('pop_bucket')['hit_top10'].mean()


## Attempted fix 2: blend in brand/category popularity for cold items

Same idea as fix 1, but using brand- and category-level training popularity as the fallback signal instead of query-match.

In [9]:
# Alternative fallback: for cold items, blend in brand- and category-level
# popularity instead of query_match_count. These capture "this item's brand/
# category is generally popular" even when the item itself has zero history,
# which is different information from what the models already saw during training.
def add_pop_fallback_score(df, model_score_col):
    model_rank = df[model_score_col].rank(pct=True)
    brand_rank = df['brand_pop_train'].rank(pct=True)
    cate_rank = df['cate4_pop_train'].rank(pct=True)
    cold_fallback = 0.5 * brand_rank + 0.5 * cate_rank
    blended = np.where(COLD_MASK, 0.5 * model_rank + 0.5 * cold_fallback, model_rank)
    return blended

test_df['bst_pop_fallback_score'] = add_pop_fallback_score(test_df, 'bst_score')
test_df['lgbm_pop_fallback_score'] = add_pop_fallback_score(test_df, 'lgbm_score')

print("Cold-start recall, brand/category popularity fallback:")
print("\nBST (with brand/cate fallback):");      print(recall_at_10_by_bucket(test_df, 'bst_pop_fallback_score'))
print("\nLightGBM (with brand/cate fallback):"); print(recall_at_10_by_bucket(test_df, 'lgbm_pop_fallback_score'))

Cold-start recall, brand/category popularity fallback:

BST (with brand/cate fallback):


/tmp/ipykernel_501/2206598482.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return relevant.groupby('pop_bucket')['hit_top10'].mean()


pop_bucket
0 (no history)    0.223806
1                 0.733243
2-5               0.858941
6-20              0.958275
20+               0.985808
Name: hit_top10, dtype: float64

LightGBM (with brand/cate fallback):
pop_bucket
0 (no history)    0.241370
1                 0.774865
2-5               0.862349
6-20              0.956421
20+               0.982533
Name: hit_top10, dtype: float64


/tmp/ipykernel_501/2206598482.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return relevant.groupby('pop_bucket')['hit_top10'].mean()


## Root-cause investigation: are cold items concentrated in different categories?

Checking whether fully cold items (no training-split history) differ systematically from items with history, in terms of which categories they belong to.

In [10]:
# Compare category distribution: cold items (no training history) vs seen items
cold_items = test_df[test_df['pop_bucket'] == '0 (no history)']
seen_items = test_df[test_df['pop_bucket'] != '0 (no history)']

for col in ['cate_id_1', 'cate_id_2', 'cate_id_3', 'cate_id_4']:
    cold_dist = cold_items[col].value_counts(normalize=True).head(5)
    seen_dist = seen_items[col].value_counts(normalize=True).head(5)
    print(f"--- {col} ---")
    print("Top categories among COLD items:")
    print(cold_dist)
    print("Top categories among SEEN items:")
    print(seen_dist)
    print()

# Also check: are there categories that appear almost ONLY among cold items
# (i.e. categories with little/no training exposure)?
cold_only_cate3 = set(cold_items['cate_id_3'].dropna().unique()) - set(seen_items['cate_id_3'].dropna().unique())
print(f"Number of cate_id_3 values seen ONLY in cold items: {len(cold_only_cate3)}")

--- cate_id_1 ---
Top categories among COLD items:
cate_id_1
920973      0.092777
63959387    0.077171
98371044    0.071730
68949654    0.064638
98719916    0.061446
Name: proportion, dtype: float64
Top categories among SEEN items:
cate_id_1
63959387    0.133605
98719916    0.127493
920973      0.118541
98371044    0.073853
68949654    0.050975
Name: proportion, dtype: float64

--- cate_id_2 ---
Top categories among COLD items:
cate_id_2
62502325    0.032023
92351230    0.026802
71940523    0.026568
33276684    0.026323
87958       0.025458
Name: proportion, dtype: float64
Top categories among SEEN items:
cate_id_2
62502325    0.105322
87958       0.046428
98427937    0.045303
11977840    0.039358
56269062    0.025779
Name: proportion, dtype: float64

--- cate_id_3 ---
Top categories among COLD items:
cate_id_3
10287851    0.031992
72526272    0.010742
52597772    0.009436
64184092    0.008108
21386661    0.007904
Name: proportion, dtype: float64
Top categories among SEEN items:
cate_i